# Survival Analysis Self-learning

## Dataset
We'll use the AIDS Clinical Trial dataset (`sksurv.datasets.load_aids`). This dataset contains information about AIDS patients in a clinical trial, with time-to-event data for progression to AIDS.

[DATASET DESCRIPTION](https://web.archive.org/web/20170517080800/http://www.umass.edu/statdata/statdata/data/actg320.txt)

In [ ]:
!pip install scikit-survival lifelines
!pip install xgboost

In [ ]:
from sksurv.datasets import load_aids
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from sksurv.ensemble import RandomSurvivalForest
from sklearn.inspection import permutation_importance
import xgboost as xgb

from sksurv.util import Surv
from sklearn.model_selection import train_test_split

In [ ]:
# Load dataset
data_x, data_y = load_aids()
data = pd.DataFrame(data_x)
data['time'] = data_y['time']
data['event'] = data_y['censor']

In [ ]:
# Display the first few rows to understand the columns
display(data.head())

# Task 1: Kaplan-Meier Analysis

1.1 Overall Survival Curve

Plot the Kaplan-Meier survival curve for the entire population.

In [ ]:
# Solution
kmf = KaplanMeierFitter()
kmf.fit(data['time'], event_observed=data['event'])
kmf.plot_survival_function()
plt.title('Kaplan-Meier Estimate (Overall Population)')
plt.ylabel('Survival Probability')
plt.xlabel('Time (days)')
plt.grid(True)

1.2 Stratified Analysis

Plot separate Kaplan-Meier curves for patients in different treatment groups (use 'tx' column where 0=control, 1=treatment). Perform a log-rank test to compare the survival distributions.

In [ ]:
# Solution
# First ensure all columns have the correct data types
data['time'] = pd.to_numeric(data['time'])
data['event'] = data['event'].astype(bool)  # Convert to boolean if not already
data['tx'] = data['tx'].astype(int)  # Ensure treatment is integer

# Check for missing values
print("Missing values in time:", data['time'].isna().sum())
print("Missing values in event:", data['event'].isna().sum())
print("Missing values in tx:", data['tx'].isna().sum())

# Remove rows with missing values if any exist
data = data.dropna(subset=['time', 'event', 'tx'])

# Now plot the curves
ax = plt.subplot(111)
kmf = KaplanMeierFitter()

for tx_status in [0, 1]:
    mask = data['tx'] == tx_status
    if sum(mask) > 0:  # Only plot if there are observations
        kmf.fit(data['time'][mask],
                event_observed=data['event'][mask],
                label=f"Treatment={'Yes' if tx_status else 'No'}")
        kmf.plot_survival_function(ax=ax)

# Log-rank test
if len(data['tx'].unique()) > 1:  # Only perform test if both groups exist
    results = logrank_test(data['time'][data['tx']==1],
                          data['time'][data['tx']==0],
                          data['event'][data['tx']==1],
                          data['event'][data['tx']==0])
    results.print_summary()
else:
    print("Warning: Only one treatment group exists in the data")

plt.title('Survival by Treatment Status')
plt.tight_layout()

# Task 2: Cox Proportional Hazards Model

2.1 Fit Cox Model

Fit a Cox proportional hazards model using these covariates: ['age', 'cd4', 'karnof', 'priorzdv', 'tx']. Interpret the coefficients.

In [ ]:
# Solution
cph = CoxPHFitter()
cph.fit(data[['age', 'cd4', 'karnof', 'priorzdv', 'tx', 'time', 'event']],
       duration_col='time', event_col='event')
cph.print_summary()

# Interpretation:
# - Positive coefficients increase hazard (worse survival)
# - Negative coefficients decrease hazard (better survival)
# - exp(coef) is the hazard ratio
# Example interpretation for 'cd4':
# For each unit increase in CD4 count, the hazard of progression decreases by X%

2.2 Partial Effects

Plot the partial effects of 'cd4' (CD4 count) on survival probability.

In [ ]:
# Generate sensible values for cd4
cd4_values = np.linspace(
    max(data['cd4'].quantile(0.05), 10),  # Ensures minimum of 10 cells/mm³
    data['cd4'].quantile(0.95),
    6
).astype(int)

# Create the partial effects plot
plt.figure(figsize=(10, 6))
cph.plot_partial_effects_on_outcome(
    covariates='cd4',
    values=cd4_values,
    cmap='coolwarm',
    plot_baseline=False
)

# Add plot formatting
plt.title('Partial Effect of CD4 Count on Survival Probability\n(AIDS Clinical Trial Data)')
plt.xlabel('Time Since Treatment Initiation (days)')
plt.ylabel('Survival Probability')
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(title='CD4 Count (cells/mm³)')

plt.tight_layout()
plt.show()

# Print the CD4 values used for reference
print("CD4 values used in analysis:", cd4_values)
print(f"Full CD4 range in dataset: {data['cd4'].min():.1f} to {data['cd4'].max():.1f} cells/mm³")

# Task 3: Random Survival Forest

3.1 RSF Model

Fit RSF Model and Predict Survival Functions

In [ ]:
# Prepare data
y = Surv.from_arrays(data['event'], data['time'])
X = data.drop(['time', 'event'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Fit model
rsf = RandomSurvivalForest(n_estimators=1000, min_samples_split=10,
                          min_samples_leaf=15, random_state=42)
rsf.fit(X_train, y_train)

# Select high and low CD4 count patients for comparison
X_test_high = X_test[X_test['cd4'] > X_test['cd4'].median()].sample(10, random_state=42)
X_test_low = X_test[X_test['cd4'] <= X_test['cd4'].median()].sample(10, random_state=42)
X_test_sel = pd.concat([X_test_high, X_test_low])

# Predict survival functions
surv = rsf.predict_survival_function(X_test_sel, return_array=True)
high_cd4_mean = surv[:10, :].mean(axis=0)
low_cd4_mean = surv[10:, :].mean(axis=0)

# Plot results
plt.figure(figsize=(10, 6))
plt.step(rsf.unique_times_, high_cd4_mean, where='post', label='High CD4 (>median)')
plt.step(rsf.unique_times_, low_cd4_mean, where='post', label='Low CD4 (≤median)')
plt.title('Predicted Survival by CD4 Count')
plt.ylabel('Survival Probability')
plt.xlabel('Time (days)')
plt.legend()
plt.grid(True)

3.2 Feature Importance Analysis

In [ ]:
result = permutation_importance(rsf, X_test, y_test, n_repeats=15, random_state=42)

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': result['importances_mean'],
    'importance_std': result['importances_std']
}).sort_values('importance_mean', ascending=False)

importance_df.plot.barh(x='feature', y='importance_mean',
                       xerr='importance_std', legend=False)
plt.title('Permutation Importance')
plt.xlabel('Importance Score')
plt.tight_layout()

# Task 4: XGBoost AFT Model

4.1 Fit AFT Model

Fit an Accelerated Failure Time (AFT) model using XGBoost with the same covariates as the Cox model.

In [ ]:
# Solution
# Prepare data for XGBoost AFT
dtrain = xgb.DMatrix(X_train[['age', 'cd4', 'karnof', 'priorzdv', 'tx']],
                     enable_categorical=True)
dtrain.set_float_info('label_lower_bound', y_train['time'])
dtrain.set_float_info('label_upper_bound', np.where(
    y_train['event'], y_train['time'], np.inf))

dtest = xgb.DMatrix(X_test[['age', 'cd4', 'karnof', 'priorzdv', 'tx']],
                    enable_categorical=True)
dtest.set_float_info('label_lower_bound', y_test['time'])
dtest.set_float_info('label_upper_bound', np.where(
    y_test['event'], y_test['time'], np.inf))

# Parameters
params = {
    'objective': 'survival:aft',
    'eval_metric': 'aft-nloglik',
    'aft_loss_distribution': 'normal',
    'aft_loss_distribution_scale': 1.0,
    'tree_method': 'hist',
    'learning_rate': 0.1,
    'max_depth': 3
}

# Train model
bst = xgb.train(params, dtrain, num_boost_round=100,
                evals=[(dtrain, 'train'), (dtest, 'test')],
                early_stopping_rounds=10, verbose_eval=10)

Plot feature importance from the XGBoost model.

In [ ]:
# Solution
xgb.plot_importance(bst)
plt.title('XGBoost AFT Feature Importance')
plt.tight_layout()

# Task 5: Model Comparison

Compare the performance of the RSF, and XGBoost AFT models using concordance index.

In [ ]:
# Be carefull, all models default outputs are different (partial_hazards, expected survival time, risk_scores)
# But we still compare performance without additional transformations of predicts
# Cause we compare not absolute values, but rankings

In [ ]:
# Solution
from sksurv.metrics import concordance_index_censored

# Attention! Cox model was trained on full data
# # Cox model predictions
cph_pred = cph.predict_partial_hazard(X_test)

# RSF predictions
rsf_pred = rsf.predict(X_test)

# XGBoost predictions
xgb_pred = bst.predict(dtest)

# Calculate C-index
cph_cindex = concordance_index_censored(y_test['event'], y_test['time'], cph_pred[X_test.index])[0]

# For rsf C-index can be calculated by # rsf_cindex = rsf.score(X_test, y_test)
rsf_cindex = concordance_index_censored(y_test['event'], y_test['time'], rsf_pred)[0]
xgb_cindex = concordance_index_censored(y_test['event'], y_test['time'], -xgb_pred)[0] # change sign of prediction, greater time means smaller risk

print(f"Cox PH C-index: {cph_cindex:.3f}")
print(f"RSF C-index: {rsf_cindex:.3f}")
print(f"XGBoost AFT C-index: {xgb_cindex:.3f}")